## 1. SlidesGo Template

In [ ]:
import os
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
import pandas as pd
import time

CSV_PATH = r"C:\Users\naeun\capstone\data\slidescarnival_dataset.csv"


def load_existing_titles(csv_path=CSV_PATH):
    """기존 CSV의 title 목록을 set으로 로드"""
    if not os.path.exists(csv_path):
        return set()
    try:
        df = pd.read_csv(csv_path)
        return set(df["title"].astype(str).tolist())
    except Exception:
        return set()


def scrape_slidescarnival_selenium(start_page=1, end_page=3, delay=2):
    options = webdriver.ChromeOptions()
    options.binary_location = r"C:/Program Files/Google/Chrome/Application/chrome.exe"
    options.add_argument('--headless')
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')

    service = Service(r"C:/Users/naeun/capstone/chromedriver-win64/chromedriver.exe")
    driver = webdriver.Chrome(service=service, options=options)

    existing_titles = load_existing_titles()
    print(f"[INFO] Loaded {len(existing_titles)} existing titles.")

    all_data = []

    for page in range(start_page, end_page + 1):
        url = f"https://www.slidescarnival.com/category/free-templates/powerpoint-templates/page/{page}"
        print(f"\nScraping page {page} … URL: {url}")
        driver.get(url)
        time.sleep(delay)

        cards = driver.find_elements(By.CSS_SELECTOR, "div.image-wrapper._16_9-landscape")
        print(f"[INFO] Found {len(cards)} cards on page {page}")

        for card in cards:
            try:
                template_url = card.get_attribute("data-url")
                driver.get(template_url)
                time.sleep(1)

                # ✔ 템플릿 제목은 H1 태그에서 추출
                title = driver.find_element(By.TAG_NAME, "h1").text.strip()

                # ✔ 중복이면 skip
                if title in existing_titles:
                    print(f"[SKIP] Already exists → {title}")
                    driver.back()
                    continue

                slide_imgs = driver.find_elements(By.CSS_SELECTOR, "ul.slider-container li img")
                img_urls = [
                    img.get_attribute("src")
                    for img in slide_imgs
                    if img.get_attribute("src") and img.get_attribute("src").endswith((".jpg", ".png"))
                ]

                canva_link = driver.find_element(By.CSS_SELECTOR, "a.sc-canva").get_attribute("href")
                ppt_link = driver.find_element(By.CSS_SELECTOR, "a.sc-powerpoint").get_attribute("href")
                google_link = driver.find_element(By.CSS_SELECTOR, "a.sc-googleslides").get_attribute("href")

                all_data.append({
                    "title": title,
                    "template_url": template_url,
                    "slide_imgs": img_urls,
                    "canva": canva_link,
                    "ppt": ppt_link,
                    "google": google_link
                })

                print(f"[NEW] {title}")
                existing_titles.add(title)

                driver.back()
                time.sleep(1)

            except Exception as e:
                print("[ERROR] Skipped a template:", e)
                try:
                    driver.back()
                except:
                    pass
                continue

    driver.quit()
    return all_data


def save_to_csv_append(data_list, csv_path=CSV_PATH):
    """새로운 데이터만 CSV에 append"""
    if not data_list:
        print("⚠️ No new data to save.")
        return

    new_df = pd.DataFrame(data_list)

    if os.path.exists(csv_path):
        old_df = pd.read_csv(csv_path)
        final_df = pd.concat([old_df, new_df], ignore_index=True)
    else:
        final_df = new_df

    final_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"✅ Saved total {len(final_df)} rows to {csv_path}")


if __name__ == "__main__":
    data = scrape_slidescarnival_selenium(start_page=1, end_page=15, delay=2)
    print("Newly collected:", len(data))
    save_to_csv_append(data)


Scraping page 1 … URL: https://www.slidescarnival.com/category/free-templates/powerpoint-templates?page=1
Found 9 cards on page 1
Scraping page 2 … URL: https://www.slidescarnival.com/category/free-templates/powerpoint-templates?page=2
Found 9 cards on page 2
Scraping page 3 … URL: https://www.slidescarnival.com/category/free-templates/powerpoint-templates?page=3
Found 9 cards on page 3
Total collected: 27
✅ Saved 27 samples to slidescarnival_dataset.csv


In [ ]:
import pandas as pd
import requests
from pathlib import Path
import ast
import re

csv_path = r"C:\Users\naeun\capstone\data\slidescarnival_dataset.csv"
save_dir = Path(r"C:\Users\naeun\capstone\data\images")
save_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(csv_path)

# --- 기존 이미지 파일 목록에서 제목만 비교하도록 준비 ---
existing_files = list(save_dir.glob("*.jpg"))

def normalize_filename(name: str):
    """
    ex) '0_Blue-Green-and-Red-Professional-Consulting-Pitch-Deck-1.jpg'
        → 'Blue-Green-and-Red-Professional-Consulting-Pitch-Deck'
    """
    name = name.replace(".jpg", "")
    name = re.sub(r"^\d+_", "", name)  # 앞 번호_ 제거
    name = re.sub(r"[-_]\d+$", "", name)  # 뒤 -1, _1 제거
    return name.lower()

normalized_existing = [normalize_filename(f.name) for f in existing_files]

START_INDEX = 60
for idx, row in df.iterrows():
    if idx < START_INDEX:
        continue
    title = str(row["title"]).strip()
    title_norm = title.lower()

    # 이 제목이 포함된 기존 파일이 있으면 skip
    if any(title_norm in ex for ex in normalized_existing):
        print(f"[SKIP] 이미 '{title}' 관련 이미지 존재함. → 다운로드 생략")
        continue

    # slide_imgs 문자열을 리스트로 변환
    try:
        img_list = ast.literal_eval(row["slide_imgs"])
    except:
        print(f"[ERR] slide_imgs 파싱 실패: {title}")
        continue

    # 이미지 여러 개 다운로드
    for i, img_url in enumerate(img_list):
        save_path = save_dir / f"{title}_slide_{i+1}.jpg"

        try:
            print(f"[DOWNLOAD] {save_path.name} 다운로드 중…")
            r = requests.get(img_url, timeout=15)
            r.raise_for_status()

            with open(save_path, "wb") as f:
                f.write(r.content)

            print(f"[OK] {save_path.name} 저장 완료.")

        except Exception as e:
            print(f"[ERR] {save_path.name} 다운로드 실패: {e}")


# 라벨링

In [1]:
import gradio as gr
import os
import json
import random

# ===============================
# 기본 경로 설정
# ===============================
DATA_DIR = r"C:\Users\naeun\capstone\data"
IMAGE_DIR = os.path.join(DATA_DIR, "images")
LABEL_DIR = os.path.join(DATA_DIR, "labels")
os.makedirs(LABEL_DIR, exist_ok=True)

# 이미지 목록 불러오기 (jpg/png 지원)
image_files = sorted([
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith((".jpg", ".png"))
])

# ===============================
# 샘플링: human labeling용 초기 샘플
# ===============================
NUM_INITIAL_LABEL = 300
initial_sample = random.sample(image_files, min(NUM_INITIAL_LABEL, len(image_files)))

# ===============================
# JSON 저장 + 다음 이미지 선택
# ===============================
def save_label_and_next(selected_image, layout, text_amount, color_contrast, visuals, consistency, feedback_text):
    if not selected_image:
        return gr.update(value=None), "❌ 이미지가 선택되지 않았습니다."

    json_path = os.path.join(LABEL_DIR, selected_image.rsplit(".",1)[0] + ".json")
    ppt_name = selected_image.rsplit(".",1)[0] + ".pptx"

    # 객체 피드백 파싱
    try:
        object_feedback = json.loads(feedback_text) if feedback_text.strip() else []
    except json.JSONDecodeError:
        return gr.update(value=selected_image), "❌ 객체 피드백 JSON 형식 오류!"

    data = {
        "filename": ppt_name,
        "overall_scores": {
            "layout": layout,
            "text_amount": text_amount,
            "color_contrast": color_contrast,
            "visuals": visuals,
            "consistency": consistency,
        },
        "object_feedback": object_feedback
    }

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    # 자동 next
    next_img = get_next_image(selected_image)
    return gr.update(value=next_img), f"✅ '{selected_image}' 라벨 저장 완료!"

# ===============================
# 다음 이미지 선택
# ===============================
def get_next_image(current):
    idx = initial_sample.index(current) if current in initial_sample else 0
    if idx + 1 < len(initial_sample):
        return initial_sample[idx + 1]
    return initial_sample[0]  # 마지막이면 처음으로

# ===============================
# 이미지 로딩 (bytes)
# ===============================
from PIL import Image

def load_image(selected_image):
    if not selected_image:
        return None
    path = os.path.join(IMAGE_DIR, selected_image)
    # 1️⃣ 경로 그대로 반환
    return path
    # 2️⃣ 또는 PIL.Image로 열어서 반환 가능
    # return Image.open(path)

# ===============================
# Gradio UI
# ===============================
with gr.Blocks() as demo:
    gr.Markdown("# 🧩 Semi-Automatic 슬라이드 라벨링 툴")

    with gr.Row():
        selected_image = gr.Dropdown(
            choices=initial_sample,
            label="🎞 이미지 선택",
            value=initial_sample[0] if initial_sample else None
        )
        image_display = gr.Image(label="슬라이드 미리보기")

    with gr.Row():
        layout = gr.Slider(0, 1, value=0.5, step=0.1, label="Layout")
        text_amount = gr.Slider(0, 1, value=0.5, step=0.1, label="Text Amount")
        color_contrast = gr.Slider(0, 1, value=0.5, step=0.1, label="Color Contrast")
        visuals = gr.Slider(0, 1, value=0.5, step=0.1, label="Visuals")
        consistency = gr.Slider(0, 1, value=0.5, step=0.1, label="Consistency")

    gr.Markdown("### 🧾 객체별 피드백(JSON 배열)")

    feedback_text = gr.Textbox(
        lines=6,
        placeholder='[{"object_id":"shape_12","type":"rect","position":{"x":100,"y":200,"w":300,"h":100},"issue":"contrast low","suggestion":"increase contrast"}]'
    )

    save_btn = gr.Button("💾 저장 후 다음 이미지로 →")
    output_msg = gr.Markdown()

    # 이미지 선택 시 미리보기 업데이트
    selected_image.change(load_image, inputs=selected_image, outputs=image_display)

    # 저장 + 자동 next
    save_btn.click(
        save_label_and_next,
        inputs=[selected_image, layout, text_amount, color_contrast, visuals, consistency, feedback_text],
        outputs=[selected_image, output_msg]
    )

# ===============================
# 실행
# ===============================
if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=None)


c:\Users\naeun\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.


### CLIP + Linear Probe
-> 자동 라벨링 학습

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from PIL import Image
import json
import clip

# ===============================
# 경로 설정
# ===============================
DATA_DIR = Path(r"C:\Users\naeun\capstone\data")
IMAGE_DIR = DATA_DIR / "images"
LABEL_DIR = DATA_DIR / "labels"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("🔥 device =", device)

# ===============================
# Dataset — CLIP preprocess 사용
# ===============================
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)

class SlideDataset(Dataset):
    def __init__(self, image_dir, label_dir, preprocess):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.preprocess = preprocess

        self.images = []
        for f in image_dir.iterdir():
            if f.suffix.lower() in [".jpg", ".png"]:
                if (label_dir / f"{f.stem}.json").exists():
                    self.images.append(f)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        img = Image.open(img_path).convert("RGB")
        img = self.preprocess(img)

        with open(self.label_dir / f"{img_path.stem}.json", "r", encoding="utf-8") as f:
            data = json.load(f)

        s = data["overall_scores"]
        label = torch.tensor([
            s["layout"],
            s["text_amount"],
            s["color_contrast"],
            s["visuals"],
            s["consistency"]
        ], dtype=torch.float32)

        return img, label


dataset = SlideDataset(IMAGE_DIR, LABEL_DIR, clip_preprocess)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)


# ===============================
# 안정형 Linear Probe 정의
# ===============================
class CLIPClassifier(nn.Module):
    def __init__(self, clip_model, output_dim=5):
        super().__init__()
        self.clip = clip_model

        feature_dim = clip_model.visual.output_dim

        self.probe = nn.Sequential(
            nn.LayerNorm(feature_dim),              # BN → LN (small batch 안정성)
            nn.Linear(feature_dim, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, output_dim),
            nn.Sigmoid()                           # 출력 collapse 방지 + 0~1 보장
        )

        # Xavier 초기화
        for m in self.probe:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)

        # CLIP freeze
        for p in self.clip.parameters():
            p.requires_grad = False

    def forward(self, x):
        with torch.no_grad():
            feats = self.clip.encode_image(x)      # (B, 512)
        return self.probe(feats)


model = CLIPClassifier(clip_model).to(device)


# ===============================
# Loss & Optimizer
# ===============================
criterion = nn.SmoothL1Loss()    # L1 기반 → MSE collapse 방지
optimizer = optim.AdamW(
    model.probe.parameters(),
    lr=3e-4,
    weight_decay=0.01
)

# ===============================
# Training Loop
# ===============================
epochs = 12
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for imgs, labels in dataloader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss / len(dataset):.4f}")

# ===============================
# 저장
# ===============================
torch.save(model.state_dict(), "clip_linear_probe.pth")
print("✅ 저장 완료!")


100%|███████████████████████████████████████| 338M/338M [00:36<00:00, 9.63MiB/s]


Epoch 1/10, Loss: 0.3580
Epoch 2/10, Loss: 0.0930
Epoch 3/10, Loss: 0.0783
Epoch 4/10, Loss: 0.0569
Epoch 5/10, Loss: 0.0483
Epoch 6/10, Loss: 0.0411
Epoch 7/10, Loss: 0.0363
Epoch 8/10, Loss: 0.0322
Epoch 9/10, Loss: 0.0289
Epoch 10/10, Loss: 0.0265
✅ Linear Probe 학습 완료, 저장됨!


### Auto Labeling

In [ ]:
import torch
import torch.nn as nn
from pathlib import Path
from PIL import Image
import json
import clip

device = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------------------
# 1) CLIP + Probe 로드
# -------------------------------------
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)

class CLIPClassifier(nn.Module):
    def __init__(self, clip_model, output_dim=5):
        super().__init__()
        self.clip = clip_model
        feature_dim = clip_model.visual.output_dim

        self.probe = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, output_dim),
            nn.Sigmoid()
        )

        for p in self.clip.parameters():
            p.requires_grad = False

    def forward(self, x):
        with torch.no_grad():
            feats = self.clip.encode_image(x)
        return self.probe(feats)

model = CLIPClassifier(clip_model).to(device)
state = torch.load("clip_linear_probe.pth", map_location=device)
model.load_state_dict(state)
model.eval()

print("🔥 모델 로딩 완료")

# -------------------------------------
# 2) 경로 설정
# -------------------------------------
DATA_DIR = Path(r"C:\Users\naeun\capstone\data")
IMAGE_DIR = DATA_DIR / "images"
LABEL_DIR = DATA_DIR / "labels"
LABEL_DIR.mkdir(exist_ok=True)

# -------------------------------------
# 3) Autolabeling
# -------------------------------------
def autolabel_image(img_path):
    img = Image.open(img_path).convert("RGB")
    img_tensor = clip_preprocess(img).unsqueeze(0).to(device)

    with torch.no_grad():
        preds = model(img_tensor)[0].cpu().numpy().tolist()

    # 출력: 0~1 사이 값이므로 그대로 저장하거나 scaling 가능
    scores = {
        "layout": float(preds[0]),
        "text_amount": float(preds[1]),
        "color_contrast": float(preds[2]),
        "visuals": float(preds[3]),
        "consistency": float(preds[4])
    }

    return scores


# -------------------------------------
# 4) 모든 이미지 자동 라벨링
# -------------------------------------
count = 0

for img_path in IMAGE_DIR.iterdir():
    if img_path.suffix.lower() not in [".jpg", ".png"]:
        continue

    json_path = LABEL_DIR / f"{img_path.stem}.json"

    # 이미 label이 있으면 skip
    if json_path.exists():
        continue

    scores = autolabel_image(img_path)

    output = {
        "overall_scores": scores,
        "autolabeled": True
    }

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=4, ensure_ascii=False)

    count += 1
    print(f"📌 Autolabeled → {img_path.name}")

print(f"\n🎉 Autolabeling 완료! 새로 생성된 라벨 수: {count}")


✅ 자동 라벨링 완료!
